# RSNA Knee Abnormality Detection: exploratory data analysis
---

## Scope and method

This notebook studies the anatomical, textual, sequence and label structure of the Kaggle **RSNA Knee Abnormality
Detection** data set. Every analysis step answers three questions:
1. **Method and tool:** which analysis, library and formula is used?
2. **Clinical and mathematical reasoning:** why is it done, and what does it mean in medicine and machine learning?
3. **Finding and modelling decision:** how does the result shape our model and preprocessing pipeline?

### Outline
- **Part 1:** data set structure and plane completeness
- **Part 2:** label distribution and clinical co-occurrence (disease synergy)
- **Part 3:** multilingual report analysis
- **Part 4:** MR physics: fluid-sensitive and fat-suppressed sequences
- **Part 5:** 2.5D windowing geometry and slice pooling
- **Part 6:** modelling blueprint

In [ ]:
import os
import re
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('default')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 100
})

# Paths
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = Path('../data') if (Path('../data').exists()) else Path('data')

train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_series = pd.read_csv(DATA_DIR / 'train_series.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
test_series = pd.read_csv(DATA_DIR / 'test_series.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET_COLS = [c for c in sample_sub.columns if c != 'StudyInstanceUID']
print(f"Data loaded. Number of target findings: {len(TARGET_COLS)}")

## Part 1: data set structure and plane completeness

### Method and tool
`pd.pivot_table` and `crosstab` on `train_series.csv` turn the series of every study (`StudyInstanceUID`) into a
frequency matrix by `Anatomical_Plane` (sagittal, coronal, axial).

### Clinical and mathematical reasoning
The biggest architectural difficulty of multi-series MRI problems is a **missing modality**: if a study lacks a
plane (e.g. sagittal present but no axial), findings that depend on that plane (e.g. patellofemoral OA) need
imputation or plane-agnostic architectures. So the plane completeness of the data set has to be measured.

In [ ]:
# Plane completeness matrix
plane_pivot = train_series.pivot_table(
    index='StudyInstanceUID',
    columns='Anatomical_Plane',
    values='SeriesInstanceUID',
    aggfunc='count',
    fill_value=0
)

completeness = ((plane_pivot['Sagittal'] > 0) & (plane_pivot['Coronal'] > 0) & (plane_pivot['Axial'] > 0)).mean() * 100
print(f"Studies with all three planes (sagittal + coronal + axial): {completeness:.2f}%")

# Series per study
series_per_study = train_series.groupby('StudyInstanceUID')['SeriesInstanceUID'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Share of studies per plane
plane_means = (plane_pivot > 0).mean() * 100
axes[0].bar(plane_means.index, plane_means.values, color=['#2b5c8f', '#4682b4', '#6baed6'], edgecolor='black', alpha=0.85)
axes[0].set_title('Studies covered per plane (%)', weight='bold')
axes[0].set_ylabel('Coverage (%)')
axes[0].set_ylim(0, 115)
for i, v in enumerate(plane_means.values):
    axes[0].text(i, v + 2, f"{v:.1f}%", ha='center', weight='bold')

# Histogram of series counts
axes[1].hist(series_per_study, bins=range(3, 16), color='#2e8b57', edgecolor='black', alpha=0.8, align='left')
axes[1].set_title('Series per study', weight='bold')
axes[1].set_xlabel('Series')
axes[1].set_ylabel('Studies')
axes[1].axvline(series_per_study.mean(), color='red', linestyle='--', label=f'Mean: {series_per_study.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### Finding and modelling decision
- **Completeness:** **100% of the studies have all three planes** (sagittal, coronal and axial).
- **Volume variation:** 5.53 series per study on average, at least 3 and at most 14.
- **Decision:** no time is spent on missing-modality handling; the model takes the three planes of every study as
  simultaneous inputs (multi-view fusion).

## Part 2: label distribution and clinical co-occurrence

### Method and tool
On the 58 manually verified gold studies: positive rates, the Pearson correlation matrix between labels and a
conditional-probability analysis:

$$\text{Risk Ratio} = \frac{P(\text{Finding}_j \mid \text{ACL}=1)}{P(\text{Finding}_j \mid \text{ACL}=0)}$$

### Clinical and mathematical reasoning
Knee biomechanics couples injuries. In an acute anterior cruciate ligament (ACL) tear the tibia subluxes forward and
impacts the lateral femoral condyle, which causes bone contusions and meniscal tears. Detecting this synergy tells us
whether multi-task (multi-label) learning can exploit shared structure.

In [ ]:
gold_mask = train_df['ACL'].notna()
gold_df = train_df[gold_mask].copy()
unlabeled_df = train_df[~gold_mask].copy()

print(f"Gold-labelled studies   : {len(gold_df)} ({len(gold_df)/len(train_df)*100:.2f}%)")
print(f"Report-only studies     : {len(unlabeled_df)} ({len(unlabeled_df)/len(train_df)*100:.2f}%)")

# Positive rate of the 12 findings
target_means = gold_df[TARGET_COLS].mean().sort_values(ascending=False)

plt.figure(figsize=(11, 5))
bars = plt.barh(target_means.index, target_means.values * 100, color='#1f77b4', edgecolor='black', alpha=0.85)
plt.title('Prevalence of each finding in the 58 gold studies (%)', weight='bold')
plt.xlabel('Positive rate (%)')
for i, bar in enumerate(bars):
    val = target_means.values[i] * 100
    count = int(round(val * len(gold_df) / 100))
    plt.text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.1f}% ({count}/{len(gold_df)})")
plt.xlim(0, 75)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Correlation between findings and conditional risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Correlation heatmap
corr = gold_df[TARGET_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-0.6, vmax=0.6, ax=axes[0], cbar_kws={'label': 'Pearson r'})
axes[0].set_title('Correlation between the 12 findings', weight='bold')

# 2. Risk ratio conditional on ACL
acl_pos = gold_df[gold_df['ACL'] == 1.0]
acl_neg = gold_df[gold_df['ACL'] == 0.0]
synergy_records = []

for t in TARGET_COLS:
    if t != 'ACL':
        p1 = acl_pos[t].mean()
        p0 = acl_neg[t].mean()
        ratio = p1 / (p0 + 1e-5)
        synergy_records.append({'Finding': t, 'Rate_if_ACL_pos': p1 * 100, 'Rate_if_ACL_neg': p0 * 100, 'Risk_ratio': ratio})

synergy_df = pd.DataFrame(synergy_records).sort_values(by='Risk_ratio', ascending=True)
axes[1].barh(synergy_df['Finding'], synergy_df['Risk_ratio'], color='#d95f02', edgecolor='black', alpha=0.85)
axes[1].axvline(1.0, color='black', linestyle='--', label='No effect (1.0x)')
axes[1].set_title('Risk ratio of other findings given an ACL tear', weight='bold')
axes[1].set_xlabel('Risk ratio (x)')
axes[1].legend()

plt.tight_layout()
plt.show()

display(synergy_df.sort_values(by='Risk_ratio', ascending=False).reset_index(drop=True))

### Finding and modelling decision
1. **Biomechanical link (contusion):** with an ACL tear the contusion rate rises **3.07×** (to 54.2%), matching the
   *pivot-shift bone bruise* pattern described in the literature.
2. **Secondary damage (fracture and effusion):** with an ACL tear the fracture risk is 2.23× and the effusion rate 75%.
3. **Inverse relation (osteoarthritis / medial OA):** with an ACL tear the OA rate falls to **0.35×**: ACL tears are
   acute injuries of young athletes, OA is degeneration of older patients.
- **Decision:** instead of 12 independent binary classifiers, one **shared-representation multi-label network** that
  can learn these dependencies.

## Part 3: multilingual report analysis

### Method and tool
The free-text `Report` fields of the 4,349 unlabelled studies are classified with lexical patterns and multilingual
medical term lists; text length, token counts and typical phrasings are examined.

### Clinical and mathematical reasoning
A regex or dictionary rule engine written for one language breaks on negation patterns in other languages
(*"no"*, *"sin"*, *"aucun"*, *"zonder"*, *"intact"*), which loses information and creates label noise.

In [ ]:
def classify_language(text):
    t = str(text).lower()
    if any(k in t for k in [' the ', 'normal', 'tear', 'intact', 'compartment', 'effusion']):
        return 'English'
    elif any(k in t for k in [' de ', 'rotura', 'derrame', 'rodilla', 'menisco', 'edema']):
        return 'Spanish'
    elif any(k in t for k in ['aucun', 'fracture', 'dégénératif', 'articulaire', 'droite']):
        return 'French'
    elif any(k in t for k in ['knie', 'rechts', 'inlichtingen', 'zonder', 'meniscusscheur']):
        return 'Dutch / German'
    return 'Other / mixed'

train_df['detected_lang'] = train_df['Report'].apply(classify_language)
lang_dist = train_df['detected_lang'].value_counts()

plt.figure(figsize=(10, 4.5))
bars = plt.bar(lang_dist.index, lang_dist.values, color=['#3182bd', '#6baed6', '#9ecae1', '#c6dbef', '#e6550d'], edgecolor='black')
plt.title('Language of the radiology reports', weight='bold')
plt.ylabel('Reports')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 30, f"{yval}\n({yval/len(train_df)*100:.1f}%)", ha='center', va='bottom', weight='bold')
plt.ylim(0, max(lang_dist.values) + 300)
plt.tight_layout()
plt.show()

### Sample reports in different languages

In [ ]:
for lang in ['English', 'Spanish', 'French', 'Dutch / German']:
    sample = train_df[train_df['detected_lang'] == lang]['Report'].iloc[0]
    clean_sample = ' '.join(sample.split()[:45]) + '...'
    print(f"[{lang.upper()} SAMPLE REPORT]:\n{clean_sample}\n")

### Finding and modelling decision
- **Multilingual reality:** reports are in English (43%), French (24%), Spanish (17%) and Dutch / German (16%).
- **Regex is not viable:** terms, negations and abbreviations differ per language; a rule-based method would fail.
- **Decision:** extract the 12 labels with **multilingual language models** (LLMs) or public LLM-distilled label sets.

## Part 4: MR physics, fluid-sensitive and fat-suppressed sequences

### Method and tool
Cross-frequency analysis of `Anatomical_Plane`, `Fluid_Sensitive` (T2/PD weighted) and `Fat_Suppression` (FS/STIR)
in `train_series.csv`.

### Clinical and mathematical reasoning
RF pulse sequences set the tissue contrast:
- **Fluid-sensitive:** free water is bright (hyperintense) on T2 and proton-density (PD) images; tears, joint fluid and
  inflammation are diagnosed on these series.
- **Fat suppression:** the fat signal of the bone marrow is nulled, so oedema and contusions inside the bone show up.

In [ ]:
fs_ct = pd.crosstab(
    train_series['Anatomical_Plane'],
    [train_series['Fluid_Sensitive'], train_series['Fat_Suppression']],
    rownames=['Anatomical plane'],
    colnames=['Fluid_Sensitive', 'Fat_Suppression']
)

print("Fluid-sensitive and fat-suppressed series per plane:")
display(fs_ct)

# Mean number of fluid-sensitive series per study
fs_study_avg = train_series.groupby('StudyInstanceUID')['Fluid_Sensitive'].sum().mean()
print(f"\nMean fluid-sensitive & fat-suppressed series per study: {fs_study_avg:.2f}")

### Finding and modelling decision
- Every study has on average **3.2 fluid-sensitive, fat-suppressed series**.
- **Decision:** feed the sequence type to the model (as a slot / sequence embedding), especially for `Effusion`,
  `Synovitis` and `Contusion`.

## Part 5: 2.5D windowing geometry and slice pooling

### Method and tool
Three consecutive slices of a 3D MR series, $[S_{i-1}, S_i, S_{i+1}]$, are stacked as RGB channels after ordering the
slices by their physical position along the slice normal.

### Clinical and mathematical reasoning
- **Why not pure 3D?** 3D convolutions explode the parameter count and memory, and cannot use ImageNet-pretrained weights.
- **Why not pure 2D?** 2D models miss how a lesion continues from slice to slice.
- **Why not average pooling?** A meniscal tear may be visible on only **2 or 3 slices** of a 30-slice series. Averaging
  slice features dilutes the tear signal among healthy slices, so **finding-specific attention pooling** is needed.

In [ ]:
def simulate_25d_stack(num_slices=32, target_windows=20):
    """
    Drop the noisy edge slices of a 32-slice MR volume and build consecutive
    three-slice RGB windows.
    """
    # Drop the edge slices (10% at each end)
    start = int(num_slices * 0.10)
    end = int(num_slices * 0.90)
    core_slices = list(range(start, end))
    
    # Three-slice windows
    windows = []
    for i in range(len(core_slices) - 2):
        windows.append((core_slices[i], core_slices[i+1], core_slices[i+2]))
        
    return core_slices, windows

core, windows = simulate_25d_stack(num_slices=32)
print(f"Raw slices              : 32")
print(f"Core anatomical slices  : {len(core)} (indices {core[0]} - {core[-1]})")
print(f"2.5D RGB windows        : {len(windows)}")
print(f"First three windows     : {windows[:3]}")

## Part 6: modelling blueprint

| area | finding | why it matters | modelling decision |
| :--- | :--- | :--- | :--- |
| **Plane completeness** | 100% of the studies have sagittal + coronal + axial | no missing-modality handling needed | **multi-view fusion** of the three planes |
| **Clinical synergy** | an ACL tear raises the contusion rate 3.07× and lowers OA | independent models miss biomechanical links | a shared **multi-label backbone** |
| **Report languages** | English 43%, French 24%, Spanish 17%, Dutch 16% | rules / regex break down | **multilingual LLM** labels |
| **MR sequences** | 3.2 fat-suppressed fluid series per study | effusion and contusion are bright on them | sequence type as an input (slot embedding) |
| **Image representation** | lesions are visible on only 2–3 slices | average pooling drowns the lesion signal | **2.5D RGB windows + 12-head attention pooling** |

---